# Praktikum 8 - **RAG - Peraturan Akademik ITB**
Nicholas Reymond Sihite - 13522144 <br/>
Catatan: pastikan runtime Colab memakai GPU

## **Import Library**

In [1]:
!pip install -q \
    "langchain" \
    "langchain-core" \
    "langchain-community" \
    "langchain-text-splitters" \
    "langchain-experimental" \
    "transformers" \
    "sentence-transformers" \
    "faiss-cpu" \
    "accelerate"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.7/358.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.4/284.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 20.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
     

In [7]:
# Unsloth ajala aowkoawk
from unsloth import FastLanguageModel

import json
import textwrap
import torch
from google.colab import files

# LangChain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Text splitter (Chunking)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

# Embeddings dan Vector Store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LLM dari HuggingFace
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from sentence_transformers import CrossEncoder
from typing import List


## **Load Data**

In [4]:
# Upload file secara manual dari laptop
uploaded = files.upload()  # pilih peraturan_akademik_itb.json

Saving peraturan_akademik_itb.json to peraturan_akademik_itb.json


In [5]:
# Load Data Peraturan Akademik ITB

DATA_PATH = "peraturan_akademik_itb.json"

def load_peraturan_as_documents(path: str):
    """
    Membaca file JSON Peraturan Akademik ITB dan mengubahnya
    menjadi list Document LangChain.
    """
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    docs = []
    for row in raw:
        bab = row.get("bab", "")
        pasal = row.get("pasal", "")
        content_list = row.get("content", [])

        # Gabungkan list kalimat menjadi satu teks panjang
        text = "\n".join(content_list)

        metadata = {
            "bab": bab,
            "pasal": pasal,
            "source": f"{bab} – {pasal}",
        }

        docs.append(Document(page_content=text, metadata=metadata))

    return docs

raw_documents = load_peraturan_as_documents(DATA_PATH)

print(f"Jumlah dokumen awal: {len(raw_documents)}")
print("Contoh 1 dokumen:")
print("Metadata:", raw_documents[0].metadata)
print(textwrap.shorten(raw_documents[0].page_content, width=300, placeholder="..."))


Jumlah dokumen awal: 72
Contoh 1 dokumen:
Metadata: {'bab': 'Peraturan Akademik Institut Teknologi Bandung', 'pasal': 'Menimbang', 'source': 'Peraturan Akademik Institut Teknologi Bandung – Menimbang'}
Bahwa telah terbit Peraturan Rektor ITB Nomor 609/IT1.A/PER/2021 tentang Peraturan Akademik Institut Teknologi Bandung Perguruan Tinggi Negeri Badan Hukum; Bahwa seiring dengan dinamika dan perkembangan ketentuan akademik yang berlaku, dipandang perlu melakukan penyesuaian dan penyempurnaan...


## **Load Model Fine-tuned** (bonus)

In [8]:
# Fine-tuned modelnya bisa di-download di sini ya kakak2 asisten:
# https://drive.google.com/drive/folders/1YC4WmXh5Zf328WDed7WYb7Jda_Ttp4lB?usp=sharing
MODEL_NAME = "indoqa-lora-1b"  # Upload dulu folder-nya ke files Colab yak wkwk
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500,
    temperature=0.1,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    return_full_text=False,
    stopping_criteria=None,
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Unsloth 2025.11.4 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Device set to use cuda:0
/tmp/ipython-input-1845669458.py:28: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generation_pipeline)


## **Indexing**

### Load Embedding Model

In [10]:
# Pakai model embedding umum dan ringan
# EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},  # cocok untuk similarity search
)

/tmp/ipython-input-1962753681.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Chunking

In [38]:
# Parameter chunk bisa dilakukan eksperimen
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

chunked_documents = text_splitter.split_documents(raw_documents)

print(f"Jumlah dokumen setelah chunking: {len(chunked_documents)}")
print("Contoh 1 chunk:")
print("Metadata:", chunked_documents[0].metadata)
print(textwrap.shorten(chunked_documents[0].page_content, width=300, placeholder="..."))

Jumlah dokumen setelah chunking: 182
Contoh 1 chunk:
Metadata: {'bab': 'Peraturan Akademik Institut Teknologi Bandung', 'pasal': 'Menimbang', 'source': 'Peraturan Akademik Institut Teknologi Bandung – Menimbang'}
Bahwa telah terbit Peraturan Rektor ITB Nomor 609/IT1.A/PER/2021 tentang Peraturan Akademik Institut Teknologi Bandung Perguruan Tinggi Negeri Badan Hukum; Bahwa seiring dengan dinamika dan perkembangan ketentuan akademik yang berlaku, dipandang perlu melakukan penyesuaian dan penyempurnaan...


### Vectorizing and Storing

In [39]:
# Vectorizing: Embedding ke vektor

# Ubah setiap chunk menjadi vektor dan bangun index FAISS (in-memory vectordb)
vectorstore = FAISS.from_documents(chunked_documents, embeddings)

print("Vector store berhasil dibuat.")

Vector store berhasil dibuat.


### Retrieving Demo

In [55]:
retriever = vectorstore.as_retriever(
    search_type="similarity",     # bisa diubah ke "mmr" oleh praktikan sebagai bahan eksperimen
    search_kwargs={"k": 4},       # bisa diubah sebagai bahan eksperimen
)

In [56]:
# Mencoba Retrieve hasil indexing

# Coba ambil beberapa dokumen terkait query sederhana
test_query = "Apa yang dimaksud dengan Indeks Prestasi Semester (IPS) di ITB?"
results = retriever.invoke(test_query)

print(f"Jumlah dokumen hasil retrieval: {len(results)}")
for i, doc in enumerate(results, 1):
    print("=" * 40)
    print(f"Dokumen ke-{i}")
    print("Source:", doc.metadata.get("source"))
    print(textwrap.shorten(doc.page_content, width=250, placeholder="..."))

Jumlah dokumen hasil retrieval: 4
Dokumen ke-1
Source: BAB VII - Evaluasi Dan Prestasi Akademik – Pasal 40 - Indeks Prestasi Semester dan Indeks Prestasi Kumulatif
ITB menentukan prestasi akademik mahasiswa melalui Indeks Prestasi Semester (IPS) dan Indeks Prestasi Kumulatif (IPK). Indeks Prestasi Semester (IPS) sebagaimana dimaksud pada ayat (1) merupakan prestasi akademik mahasiswa yang dicapai pada setiap...
Dokumen ke-2
Source: BAB VII - Evaluasi Dan Prestasi Akademik – Pasal 40 - Indeks Prestasi Semester dan Indeks Prestasi Kumulatif
Indeks Prestasi Kumulatif (IPK) sebagaimana dimaksud pada ayat (1) merupakan prestasi akademik mahasiswa yang dicapai dalam kurun waktu tertentu atas dasar perhitungan perolehan nilai akhir sejumlah mata kuliah, dimana jika ada mata kuliah yang...
Dokumen ke-3
Source: BAB VI - Layanan Akademik – Pasal 33 - Pelaksanaan Ujian
Mahasiswa mengikuti ujian sesuai dengan jadwal dan tempat yang ditentukan oleh ITB. Masa ujian pada setiap semester tercantum pad

## **Chaining**

In [42]:
RAG_PROMPT_TEMPLATE = """Berdasarkan konteks berikut, jawab pertanyaan yang diajukan.

### Konteks:
{context}

### Pertanyaan:
{question}

### Jawaban:
"""

prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

In [57]:
# Helper function untuk format dokumen
def format_docs(docs):
    chunks = []
    for doc in docs:
        header = f"[{doc.metadata.get('source', 'tanpa sumber')}]"
        body = doc.page_content
        chunks.append(header + "\n" + body)
    return "\n\n-----\n\n".join(chunks)

# Build RAG Chain using LCEL
rag_chain = (
    RunnableParallel({
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm
    | StrOutputParser()
)

## **Testing RAG Chain**

def ask_rag_lcel(question: str, show_sources: bool = True, show_full_prompt: bool = True, debug: bool = False):
    """
    Pipeline RAG menggunakan LCEL.
    """
    # Retrieve dokumen untuk ditampilkan sebagai sumber
    retrieved_docs = retriever.invoke(question)

    # Build context untuk tampilan prompt
    if show_full_prompt:
        context = format_docs(retrieved_docs)
        full_prompt = RAG_PROMPT_TEMPLATE.format(
            context=context,
            question=question
        )
        print("**📜 Prompt yang dikirim ke LLM:**\n")
        print(full_prompt.strip())
        print("\n" + "=" * 80 + "\n")

    # Tampilkan pertanyaan
    print(f"**🧑‍🎓 Pertanyaan:**\n{question}")
    print("\n" + "=" * 80 + "\n")

    # Invoke RAG chain (LCEL)
    answer = rag_chain.invoke(question)

    # DEBUG: Show raw output before parsing
    if debug:
        print("**🔧 RAW OUTPUT (before parsing):**")
        print(repr(answer))
        print("\n" + "=" * 80 + "\n")

    # Tampilkan jawaban
    print("**🤖 Jawaban RAG:**\n")
    print(textwrap.fill(answer.strip(), width=100))
    print("\n" + "=" * 80 + "\n")

    # Tampilkan sumber
    if show_sources:
        print("**📚 Sumber konteks yang dipakai:**")
        for i, doc in enumerate(retrieved_docs, 1):
            print("-" * 80)
            print(f"[{i}] {doc.metadata.get('source')}")
            print(textwrap.shorten(doc.page_content, width=400, placeholder="..."))

    return {
        "question": question,
        "answer": answer,
        "source_documents": retrieved_docs,
    }

## **Demo**

### Q1: Benar, tetapi kurang lengkap. Asumsi karena PT-model di-fine-tune untuk memberikan jawaban singkat.

In [58]:
q1_result = ask_rag_lcel(
    "Apa saja persyaratan cum laude Magister?",
    show_sources=False,
    show_full_prompt=False,
    debug=False
)

**🧑‍🎓 Pertanyaan:**
Apa saja persyaratan cum laude Magister?


**🤖 Jawaban RAG:**

Kualitatif: Menunjukkan penghayatan yang baik tentang hakikat dan norma-norma masyarakat akademik,
serta tidak pernah mendapatkan sanksi tertulis dari ITB, baik sanksi akademik maupun non-akademik




### Q2: Salah. Asumsi: retriever memberikan dokumen yang kurang tepat karena model embedding kurang baik. Asumsi ini diambil karena setelah menggunakan model lain untuk embedding ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"), Q2 bisa dijawab dengan benar. Selain itu, teknik chunking yang digunakan juga mungkin kurang tepat karena jawabannya berada pada Bab VII Pasal 38, tetapi bagian tersebut tidak ada pada retrieved document.

---



In [59]:
q2_result = ask_rag_lcel(
    "Bagaimana representasi nilai akhir dalam bentuk angka?",
    show_sources=True,
    show_full_prompt=False,
    debug=False
)

**🧑‍🎓 Pertanyaan:**
Bagaimana representasi nilai akhir dalam bentuk angka?


**🤖 Jawaban RAG:**

huruf a dari ayat (1)


**📚 Sumber konteks yang dipakai:**
--------------------------------------------------------------------------------
[1] BAB VIII - Kelulusan – Pasal 42 - Kelulusan Program Sarjana
Jika tidak mendapatkan izin penundaan kelulusan sebagaimana dimaksud pada ayat (2), maka mahasiswa akan dipertimbangkan untuk ditetapkan dalam Rapat Yudisium ITB terdekat.
--------------------------------------------------------------------------------
[2] BAB VIII - Kelulusan – Pasal 46 - Kriteria Predikat Kelulusan untuk Program Sarjana
Pemenuhan ketentuan sebagaimana dimaksud pada huruf a dari ayat (1) diusulkan oleh Fakultas/Sekolah dan wajib mendapatkan persetujuan dari Wakil Rektor ITB yang menangani urusan akademik.
--------------------------------------------------------------------------------
[3] BAB III - Proses Pembelajaran – Pasal 12 - Sistem Kredit Semester
Bentuk pembelajaran

### Q3: Sudah Benar

In [60]:
q3_result = ask_rag_lcel(
    "Berapa jumlah semester maksimal untuk program doktor?",
    show_sources=False,
    show_full_prompt=False,
    debug=False
)

**🧑‍🎓 Pertanyaan:**
Berapa jumlah semester maksimal untuk program doktor?


**🤖 Jawaban RAG:**

10 (sepuluh)




### Q4: Sudah Benar

In [61]:
q4_result = ask_rag_lcel(
    "Pasal berapa yang mengatur tentang mahasiswa asing?",
    show_sources=False,
    show_full_prompt=False,
    debug=False
)

**🧑‍🎓 Pertanyaan:**
Pasal berapa yang mengatur tentang mahasiswa asing?


**🤖 Jawaban RAG:**

Pasal 1




### Q5: Aowkaokawok

In [ ]:
# Bonus
q5_result = ask_rag_lcel(
    "Bandingkan satuan SKS untuk program sarjana, magister, dan doktor",
    show_sources=False,
    show_full_prompt=False,
    debug=False
)

**🧑‍🎓 Pertanyaan:**
Bandingkan satuan SKS untuk program sarjana, magister, dan doktor


**🤖 Jawaban RAG:**

Program Magister dan Doktor


